In [1]:
#| default_exp core.optimizers
#| export

import numpy as np
from typing import List, Union, Optional, Dict, Any

# Import Tensor from Module 01 (now with gradient support from Module 06)
from tinytorch.core.tensor import Tensor

# Enable autograd to add gradient tracking to Tensor
# This module depends on Module 06 (Autograd) being available
from tinytorch.core.autograd import enable_autograd
enable_autograd()

# Constants for optimizer defaults
DEFAULT_LEARNING_RATE_SGD = 0.01  # Default learning rate for SGD
DEFAULT_LEARNING_RATE_ADAM = 0.001  # Default learning rate for Adam/AdamW
DEFAULT_MOMENTUM = 0.9  # Default momentum for SGD
DEFAULT_BETA1 = 0.9  # First moment decay rate for Adam
DEFAULT_BETA2 = 0.999  # Second moment decay rate for Adam
DEFAULT_EPS = 1e-8  # Small epsilon for numerical stability in Adam
DEFAULT_WEIGHT_DECAY_ADAMW = 0.01  # Default weight decay for AdamW

In [5]:
#| export
class Optimizer:
    """
    Base class for all optimizers.

    This class defines the common interface that all optimizers must implement:
    - zero_grad(): Clear gradients from parameters
    - step(): Update parameters based on gradients
    """

    def __init__(self, params: List[Tensor]):
        """
        Initialize optimizer with parameters to optimize.

        TODO: Set up the parameter list for optimization

        APPROACH:
        1. Store parameters as a list for iteration
        2. Validate that all parameters require gradients
        3. Initialize step counter for algorithms that need it

        EXAMPLE:
        >>> linear = Linear(784, 128)
        >>> optimizer = SGD(linear.parameters(), lr=0.01)

        HINT: Store parameters for iteration during optimization steps
        """
        ### BEGIN SOLUTION
        if not isinstance(params, list):
            params = list(params)

        self.params = params

        for param in self.params:
            if isinstance(param, Tensor):
                param.requires_grad = True
                param.grad = None
        self.step_count = 0
        ### END SOLUTION

    def zero_grad(self):
        """
        Clear gradients from all parameters.

        TODO: Reset all parameter gradients to None

        APPROACH:
        1. Iterate through all parameters
        2. Set each parameter's grad to None

        EXAMPLE:
        >>> optimizer.zero_grad()  # Clears all gradients
        >>> assert param.grad is None for param in optimizer.params

        WHY: Gradients accumulate by default, so we need to clear them between batches
        """
        ### BEGIN SOLUTION
        for param in self.params:
            param.grad = None
        ### END SOLUTION

    def step(self):
        """
        Update parameters based on gradients.

        This is abstract - each optimizer implements its own update rule.
        """
        raise NotImplementedError(
            f"Abstract method step() not implemented\n"
            f"  ❌ {self.__class__.__name__} inherits from Optimizer but doesn't define step()\n"
            f"  💡 Each optimizer must implement its own update rule (SGD, Adam, etc.)\n"
            f"  🔧 Override step() in your optimizer subclass:\n"
            f"      def step(self):\n"
            f"          for param in self.params:\n"
            f"              if param.grad is not None:\n"
            f"                  param.data -= self.lr * param.grad.data"
        )

In [6]:
#| export
class _ExtractGradientMixin:
    """Mixin added to Optimizer for gradient extraction."""
    def _extract_gradient(self, param: Tensor) -> np.ndarray:
        """
        Extract gradient data as a NumPy array from a parameter.

        Gradients can be stored as either a Tensor (with .data attribute)
        or as a raw NumPy array (from autograd). This helper normalizes
        both cases to a plain NumPy array for optimizer math.

        TODO: Return the gradient's underlying NumPy array

        APPROACH:
        1. Get param.grad
        2. If it's a Tensor, return its .data attribute
        3. If it's already a NumPy array, return it directly

        EXAMPLE:
        >>> param = Tensor([1.0, 2.0], requires_grad=True)
        >>> param.grad = Tensor([0.1, 0.2])
        >>> optimizer._extract_gradient(param)
        array([0.1, 0.2])

        HINT: Use isinstance(grad, Tensor) to check the type
        """
        ### BEGIN SOLUTION
        grad = param.grad
        if isinstance(grad, Tensor):
            return grad.data
        else:
            return grad
        ### END SOLUTION

# Attach _extract_gradient to Optimizer so all subclasses inherit it
Optimizer._extract_gradient = _ExtractGradientMixin._extract_gradient

In [7]:
def test_unit_extract_gradient():
    """🧪 Test _extract_gradient handles Tensor and ndarray gradients."""
    print("🧪 Unit Test: Gradient Extraction...")

    param = Tensor([1.0, 2.0], requires_grad=True)
    optimizer = Optimizer([param])

    # Case 1: Gradient is a Tensor
    param.grad = Tensor([0.1, 0.2])
    grad_data = optimizer._extract_gradient(param)
    assert isinstance(grad_data, np.ndarray), "Should return ndarray from Tensor grad"
    assert np.allclose(grad_data, [0.1, 0.2])

    # Case 2: Gradient is a raw NumPy array
    param.grad = np.array([0.3, 0.4])
    grad_data = optimizer._extract_gradient(param)
    assert isinstance(grad_data, np.ndarray), "Should return ndarray from ndarray grad"
    assert np.allclose(grad_data, [0.3, 0.4])

    # Case 3: Multi-dimensional gradient
    param_2d = Tensor([[1.0, 2.0], [3.0, 4.0]], requires_grad=True)
    opt_2d = Optimizer([param_2d])
    param_2d.grad = Tensor([[0.1, 0.2], [0.3, 0.4]])
    grad_data_2d = opt_2d._extract_gradient(param_2d)
    assert grad_data_2d.shape == (2, 2)
    assert np.allclose(grad_data_2d, [[0.1, 0.2], [0.3, 0.4]])

    print("✅ Gradient extraction works correctly!")

if __name__ == "__main__":
    test_unit_extract_gradient()

🧪 Unit Test: Gradient Extraction...
✅ Gradient extraction works correctly!


In [8]:
def test_unit_optimizer_base():
    """🧪 Test base Optimizer functionality."""
    print("🧪 Unit Test: Base Optimizer...")

    # Create test parameters
    param1 = Tensor([1.0, 2.0], requires_grad=True)
    param2 = Tensor([[3.0, 4.0], [5.0, 6.0]], requires_grad=True)

    # Create optimizer first (optimizer.__init__ resets grad to None)
    optimizer = Optimizer([param1, param2])

    # Test parameter storage
    assert len(optimizer.params) == 2
    assert optimizer.params[0] is param1
    assert optimizer.params[1] is param2
    assert optimizer.step_count == 0

    # Add gradients AFTER creating optimizer to test zero_grad properly
    param1.grad = Tensor([0.1, 0.2])
    param2.grad = Tensor([[0.3, 0.4], [0.5, 0.6]])

    # Test zero_grad
    optimizer.zero_grad()
    assert param1.grad is None
    assert param2.grad is None

    # Test that optimizer accepts any tensor (no validation required)
    # Gradient tracking is handled by the autograd module
    regular_param = Tensor([1.0])
    opt = Optimizer([regular_param])
    assert len(opt.params) == 1

    print("✅ Base Optimizer works correctly!")

if __name__ == "__main__":
    test_unit_optimizer_base()

🧪 Unit Test: Base Optimizer...
✅ Base Optimizer works correctly!


In [ ]:
#| export
class SGD(Optimizer):
    """
    Stochastic Gradient Descent with momentum.

    SGD is the foundational optimization algorithm that moves parameters
    in the direction opposite to gradients. With momentum, it remembers
    previous updates to reduce oscillations and accelerate convergence.
    """

    def __init__(self, params: List[Tensor], lr: float = DEFAULT_LEARNING_RATE_SGD, momentum: float = 0.0, weight_decay: float = 0.0):
        """
        Initialize SGD optimizer.

        TODO: Set up SGD with momentum and weight decay

        APPROACH:
        1. Call parent constructor to set up parameters
        2. Store learning rate, momentum, and weight decay
        3. Initialize momentum buffers for each parameter

        EXAMPLE:
        >>> optimizer = SGD(model.parameters(), lr=0.01, momentum=0.9)

        HINTS:
        - Momentum buffers should be initialized as None
        - They'll be created lazily on first step
        """
        ### BEGIN SOLUTION
        super().__init__(params)

        self.lr = lr
        self.momentum = momentum
        self.weight_decay = weight_decay

        # Initialize momentum buffers (created lazily)
        self.momentum_buffers = [None for _ in self.params]
        ### END SOLUTION

    def has_momentum(self) -> bool:
        """
        Check if this optimizer uses momentum.

        This explicit API method replaces the need for hasattr() checks
        in checkpointing code (Module 08).

        Returns:
            bool: True if momentum is enabled (momentum > 0), False otherwise

        EXAMPLE:
            >>> optimizer = SGD(params, lr=0.01, momentum=0.9)
            >>> optimizer.has_momentum()
            True
        """
        return self.momentum > 0

    def get_momentum_state(self) -> Optional[List]:
        """
        Get momentum buffers for checkpointing.

        This explicit API method provides safe access to momentum buffers
        without using hasattr(), making the API contract clear.

        Returns:
            Optional[List]: List of momentum buffers if momentum is enabled,
                          None otherwise

        EXAMPLE:
            >>> optimizer = SGD(params, lr=0.01, momentum=0.9)
            >>> optimizer.step()  # Initialize buffers
            >>> state = optimizer.get_momentum_state()
            >>> # Later: optimizer.set_momentum_state(state)
        """
        if not self.has_momentum():
            return None
        return [buf.copy() if buf is not None else None
                for buf in self.momentum_buffers]

    def set_momentum_state(self, state: Optional[List]) -> None:
        """
        Restore momentum buffers from checkpointing.

        This explicit API method provides safe restoration of momentum state
        without using hasattr().

        Args:
            state: List of momentum buffers or None

        EXAMPLE:
            >>> optimizer = SGD(params, lr=0.01, momentum=0.9)
            >>> state = optimizer.get_momentum_state()
            >>> # Training interruption...
            >>> new_optimizer = SGD(params, lr=0.01, momentum=0.9)
            >>> new_optimizer.set_momentum_state(state)
        """
        if state is None or not self.has_momentum():
            return

        if len(state) != len(self.momentum_buffers):
            raise ValueError(
                f"Momentum state length mismatch\n"
                f"  ❌ State has {len(state)} buffers, but optimizer has {len(self.momentum_buffers)} parameters\n"
                f"  💡 Checkpoint was saved with a different model architecture or parameter count\n"
                f"  🔧 Ensure you're loading state into an optimizer with the same number of parameters:\n"
                f"      # Check parameter counts match before restoring\n"
                f"      assert len(saved_state) == len(optimizer.params)"
            )

        for i, buf in enumerate(state):
            if buf is not None:
                self.momentum_buffers[i] = buf.copy()

    def step(self):
        """
        Perform SGD update step with momentum.

        TODO: Implement SGD parameter update by composing helpers

        APPROACH:
        1. For each parameter with gradients:
           a. Extract gradient using self._extract_gradient(param)
           b. Apply weight decay if specified
           c. Update momentum buffer
           d. Update parameter using momentum

        FORMULA:
        - With weight decay: grad = grad + weight_decay * param
        - Momentum: v = momentum * v_prev + grad
        - Update: param = param - lr * v

        HINTS:
        - Skip parameters without gradients
        - Use self._extract_gradient() from the base class
        - Initialize momentum buffers on first use
        """
        ### BEGIN SOLUTION
        for i, param in enumerate(self.params):
            if param.grad is None:
                continue

            # extract gradient using shared helper
            grad_data = self._extract_gradient(param)

            # apply weight decay
            if self.weight_decay != 0:
                grad_data = grad_data + self.weight_decay * param.data

            # update momentum buffer
            if self.momentum != 0:
                if self.momentum_buffers[i] is None:
                    # initialize momentum buffer
                    self.momentum_buffers[i] = np.zeros_like(param.data)

                # update momentum: v = momentum * v_prev + grad
                self.momentum_buffers[i] = self.momentum * self.momentum_buffers[i] + grad_data
                grad_data = self.momentum_buffers[i]

                
        ### END SOLUTION